# Lesson 11 - Model Context Protocol (MCP)

Di **Model Context Protocol (MCP)** na open standard wey dey allow agents make dem sabi find and use tools, resources, and data sources for runtime. Instead of to hardcode tools inside agent, MCP dey make agents fit connect to external servers wey dey show wetin dem fit do on top demand.

For dis lesson, you go learn:
- Wetin MCP be and why e matter for agent systems
- How di MCP client-server architecture dey work
- How to build agents wey dey use MCP-style tool discovery


## Setup

**Wetin you for get ready:**
- Microsoft Foundry project wey get deployed model
- Run `az login` to do authentication


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wetin be Model Context Protocol (MCP)?

MCP dey define one standard way wey AI agents fit take find and interact wit external tools and data sources:

- **MCP Server**: E dey expose tools, resources, and prompts through one standard protocol
- **MCP Client**: Na di agent runtime wey dey connect to servers and dey find wetin capabilities dey available
- **Dynamic Discovery**: Agents no need hardcoded tools — dem dey find wetin dey available for runtime

Dis one get powerful use for build extensible agent systems wey new capabilities fit add without to change di agent code.


## How MCP Dey Work

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Di agent (MCP client) go connect to one MCP server
2. Di server go respond with list of tools wey dey available and their schemas
3. Di agent fit come call any tool wey e don find during im reasoning
4. Results go flow back through di same protocol


## Simulètin MCP Tool Discovery

Since real MCP server need make server process dey run, we go show di pattern by using `@tool` functions wey dey simulate wetin MCP-connected accommodation service go provide.

For production, dem go find dis tools dynamically from MCP server, no be to define am locally.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## How to Build Agent Wit MCP-Style Tools


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP for Production

For production environment, MCP dey enable strong patterns:

- **Dynamic tool discovery**: Agents dey connect to MCP servers dem find tools for runtime
- **Decoupled architecture**: Tool providers fit update independently from the agent
- **Cross-organization sharing**: Teams fit show capabilities through MCP servers wey any agent fit use
- **Microsoft Agent Framework support**: MAF get built-in MCP client support through `mcp` integration

To use real MCP server with MAF, you go connect through `hosted_mcp_tool()` or MCP client integration.

**Learn more:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

For dis lesson, you learn:
- **MCP** na open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** dey make agents fit discover capabilities for runtime
- MCP dey allow **extensible, decoupled agent systems** wey tools fit dey add without code changes
- Microsoft Agent Framework dey provide **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
